In [5]:
### Loading libraries ###
import pandas as pd
import sqlite3

...but how to add the specific concentrations

In [15]:
df_toxicity_info = pd.read_excel('/Users/juliakulpa/Desktop/DB_tests_mixture_rules/toxicity_1.xlsx')

df_product = pd.read_excel('/Users/juliakulpa/Desktop/DB_tests_mixture_rules/Test_for_mixture_rules_v2.xlsx')

def skin_corr_mixture_rule_c2c(df_product, df_toxicity_info):
    df_calculation = pd.merge(df_product, df_toxicity_info, on="CAS", how="left")

    # get the unique hom materials
    hom_materials = df_product["Homogenous Material"].unique().tolist()

    # save the highest value of contribution of hom mat
    df_calculation["conc_hom_mat"] = df_calculation[["min_contribution_hom_mat", "max_contribution_hom_mat"]].max(axis=1)

    def skin_irr_mixture_rating(df):
        conc_col = "conc_hom_mat"
        rating_col = "Skin, Eye, Respiratory corrosion/irritation C2C assessment"

        #copy the dataframe
        d = df.copy()

        # Ensure concentration is numeric
        d[conc_col] = pd.to_numeric(d[conc_col], errors="coerce").fillna(0)

        # Standardise colour text
        d[rating_col] = d[rating_col].astype(str).str.strip().str.upper()

        # Sums needed for the rules
        red_sum_ge_1 = d.loc[
            (d[rating_col] == "RED") & (d[conc_col] >= 1),
            conc_col
        ].sum()

        red_sum_ge_0_1_lt_1 = d.loc[
            (d[rating_col] == "RED") & (d[conc_col] >= 0.1) & (d[conc_col] < 1),
            conc_col
        ].sum()

        grey_sum_ge_0_1 = d.loc[
            (d[rating_col] == "GREY") & (d[conc_col] >= 0.1),
            conc_col
        ].sum()

        yellow_sum_ge_1 = d.loc[
            (d[rating_col] == "YELLOW") & (d[conc_col] >= 1),
            conc_col
        ].sum()

        # Rule 1: RED
        if red_sum_ge_1 >= 5:
            mixture_rating = "RED"

        # Rule 2: GREY
        elif red_sum_ge_1 < 5 and (red_sum_ge_1 + grey_sum_ge_0_1) >= 5:
            mixture_rating = "GREY"

        # Rule 3: YELLOW
        elif (
            (red_sum_ge_1 >= 1 and red_sum_ge_1 < 5)
            or
            ((10 * red_sum_ge_0_1_lt_1) + yellow_sum_ge_1 >= 1)
        ):
            mixture_rating = "YELLOW"

        # Rule 4: GREEN
        else:
            mixture_rating = "GREEN"

        return mixture_rating

    skin_corr_for_each_material = []
    # assessment for each hom mat
    for hom_material in hom_materials:
        df_calc_hom_material = df_calculation.loc[df_product["Homogenous Material"] == hom_material]
        rating = skin_irr_mixture_rating(df_calc_hom_material)
        skin_corr_for_each_material.append({
            "hom_material": hom_material,
            f"skin_corr": rating})

    skin_results_df = pd.DataFrame(skin_corr_for_each_material)
    return skin_results_df

result = skin_corr_mixture_rule_c2c(df_product, df_toxicity_info)
print(result)

  hom_material skin_corr
0            A    YELLOW
1            B    YELLOW


In [ ]:
def eye_irr_mixture_rating(df):
    conc_col = "conc_hom_mat"
    rating_col = "eye irr"

    # Make a clean working copy
    d = df.copy()

    # Ensure concentration is numeric
    d[conc_col] = pd.to_numeric(d[conc_col], errors="coerce").fillna(0)

    # Standardise colour text
    d[rating_col] = d[rating_col].astype(str).str.strip().str.upper()

    # Sums needed for the rules
    red_sum_ge_1 = d.loc[
        (d[rating_col] == "RED") & (d[conc_col] >= 1),
        conc_col
    ].sum()

    red_sum_ge_0_1_lt_1 = d.loc[
        (d[rating_col] == "RED") & (d[conc_col] >= 0.1) & (d[conc_col] < 1),
        conc_col
    ].sum()

    grey_sum_ge_0_1 = d.loc[
        (d[rating_col] == "GREY") & (d[conc_col] >= 0.1),
        conc_col
    ].sum()

    yellow_sum_ge_1 = d.loc[
        (d[rating_col] == "YELLOW") & (d[conc_col] >= 1),
        conc_col
    ].sum()

    yellow_weighted_sum = (10 * red_sum_ge_0_1_lt_1) + yellow_sum_ge_1

    # Rule 1: RED
    if red_sum_ge_1 >= 3:
        mixture_rating = "RED"

    # Rule 2: GREY
    elif red_sum_ge_1 < 3 and (red_sum_ge_1 + grey_sum_ge_0_1) >= 3:
        mixture_rating = "GREY"

    # Rule 3: YELLOW
    elif (1 <= red_sum_ge_1 < 3) or (yellow_weighted_sum >= 10):
        mixture_rating = "YELLOW"

    # Rule 4: GREEN
    else:
        mixture_rating = "GREEN"

    return mixture_rating

In [ ]:
def calculate_ATE(df_calculation,  hom_materials, ate_specification):
    """
    :param df_calculation: dataframe with CAS and their associated toxicity
    :param ate_specification: we should specify which ATE to calculate corresponding to LD50 in the df_calcuation e.g. if we want to calculate oral one we should say LD50_oral
    :param hom_materials: homogenous materials we should calculate ATE for
    :return: dataftrame with ATE for each homogenous material
    """
    # LD50 for oral/inhalation/dermal etc.
    LD50 = ate_specification

    df_calculation = df_calculation.copy()

    # save the highest value of contribution of hom mat
    df_calculation["conc_hom_mat"] = df_calculation[["min_contribution_hom_mat", "max_contribution_hom_mat"]].max(axis=1)
    # force the data to be numeric
    df_calculation[LD50] = pd.to_numeric(df_calculation[LD50], errors='coerce')

    ### Step 1. check the concentration and exclude the one below 0.1 %
    df_calculation = df_calculation[df_calculation["conc_hom_mat"]>= 0.001].copy()

    ### Step 2. exclude the values that are below 1 % and are classified as cat.4 ( np. LD50 oral > 2000)
    if LD50 == "LD50_oral":
        exclusion_value = 2000
        CLP_info = "CLP oral class"
        LD50_tox_1 = 0.5
        LD50_tox_2 = 5
        LD50_tox_3 = 100
        LD50_tox_4 = 500
    if LD50 == "LD50_dermal":
        exclusion_value = 2000
        CLP_info = "CLP dermal class"
        LD50_tox_1 = 5
        LD50_tox_2 = 50
        LD50_tox_3 = 300
        LD50_tox_4 = 1100
    if LD50 == "LC50_gas":
        exclusion_value = 20000
        CLP_info = "CLP inhalation class"
        LD50_tox_1 = 10
        LD50_tox_2 = 100
        LD50_tox_3 = 700
        LD50_tox_4 = 4500
    if LD50 == "LC50_vapour":
        exclusion_value = 20
        CLP_info = "CLP inhalation class"
        LD50_tox_1 = 0.05
        LD50_tox_2 = 0.5
        LD50_tox_3 = 3
        LD50_tox_4 = 5
    if LD50 == "LC50_dust_mist_aerosol":
        exclusion_value = 5
        CLP_info = "CLP inhalation class"
        LD50_tox_1 = 0.005
        LD50_tox_2 = 0.05
        LD50_tox_3 = 0.5
        LD50_tox_4 = 1.5

    df_calculation = df_calculation.loc[~((df_calculation["conc_hom_mat"] < 0.01) & (df_calculation[LD50] > exclusion_value))].copy()

    # add values of LD50 for chemicals that are classified in a category but do not have an LD50 value
    # tox cat 1
    df_calculation.loc[df_calculation[LD50].isna() & df_calculation[CLP_info].str.contains("Tox. 1", na=False, regex=False),LD50] = LD50_tox_1
    # tox cat 2
    df_calculation.loc[df_calculation[LD50].isna() & df_calculation[CLP_info].str.contains("Tox. 2", na=False, regex=False),LD50] = LD50_tox_2
    # tox cat 3
    df_calculation.loc[df_calculation[LD50].isna() & df_calculation[CLP_info].str.contains("Tox. 3", na=False, regex=False),LD50] = LD50_tox_3
    # tox cat 4
    df_calculation.loc[df_calculation[LD50].isna() & df_calculation[CLP_info].str.contains("Tox. 4", na=False, regex=False),LD50] = LD50_tox_4
    ### Step 3. Calculate ATE based on:

    # 100 – (∑n % chemicals unknown > 10%) / ATE mixture = ∑n % chemical is in formulation/LD50 or LC50
    # iterate for each hom mat:
    ATE_for_each_material = []

    for hom_material in hom_materials:
        df_calc_hom_material = df_calculation.loc[df_product["Homogenous Material"] == hom_material]

        # Sum of chemical with unknown LD50 that are not "not classified" in CLP

        condition_to_be_unknown_chemical = (df_calc_hom_material["conc_hom_mat"] > 0.1 ) & (df_calc_hom_material[LD50].isna() & (df_calc_hom_material[CLP_info] != "Not classified"))
        unknown_LD_class_chemicals = df_calc_hom_material.loc[condition_to_be_unknown_chemical, "CAS"].tolist()
        sum_unknown_chemicals = df_calc_hom_material.loc[condition_to_be_unknown_chemical, "conc_hom_mat"].sum()

        # calculate 100 – (∑n % chemicals unknown > 10%)
        adjusted_100 = 100 - sum_unknown_chemicals

        # n % chemical is in formulation/LD50 (PER CHEMICAL)
        df_calc_hom_material["conc_divided_by_LD50"] = df_calc_hom_material["conc_hom_mat"]/df_calculation[LD50]

        # ∑n % chemical is in formulation/LD50 (SUM ALL)
        sum_constituents = df_calc_hom_material["conc_divided_by_LD50"].sum()

        # ATE = 100 – (∑n % chemicals unknown > 10%) / ∑n % chemical is in formulation/LD50 or LC50
        ate = adjusted_100 / sum_constituents

        # ATE_for_each_material[hom_material] = round(float(ate),2)

        ATE_for_each_material.append({
                "hom_material": hom_material,
                f"ATE_based_on_{LD50}": round(float(ate),2)})

    df = pd.DataFrame(ATE_for_each_material)
    return df, unknown_LD_class_chemicals
def obtain_all_ATE_df(df_calculation, hom_materials, all_ld_50_or_lc_50_options):
    ''''
    :param df_calculation: dataframe with CAS and their associated toxicity
    :param hom_materials: homogenous materials we should calculate ATE for
    :param all_ld_50_or_lc_50_options (specify which for which LD50 (oral, demral etc.) to calculate ATE)
    :return: dataftrame with ATE (oral, dermal etc.) for each homogenous material, list of chemicals with unknown ATE but not "not classified"
     '''
    final_df = None
    for ld_50_or_lc_50 in all_ld_50_or_lc_50_options:
        df, unknown_LD_class_chemicals = calculate_ATE(df_calculation, hom_materials, ate_specification = ld_50_or_lc_50)
        if final_df is None:
            final_df = df
        else:
            final_df = final_df.merge(df, on="hom_material", how="outer")
    return final_df, unknown_LD_class_chemicals
def classify_mixture_C2C(df_with_ate, df_product, df_toxicity_info):
    df_with_ate = df_with_ate.copy()

    #needed for GREY classification
    df_calculation = pd.merge(df_product, df_toxicity_info, on="CAS", how="left")
    # get the unique hom materials
    hom_materials = df_product["Homogenous Material"].unique().tolist()
    # save the highest value of contribution of hom mat
    df_calculation["conc_hom_mat"] = df_calculation[["min_contribution_hom_mat", "max_contribution_hom_mat"]].max(axis=1)
    oral_grey = []
    inhal_grey = []
    dermal_grey = []
    for hom_material in hom_materials:
        df = df_calculation.loc[df_product["Homogenous Material"] == hom_material]
        sum_oral_grey = df.loc[df["oral toxicity C2C assessment"]=="GREY", "conc_hom_mat"].sum()
        sum_inhal_grey = df.loc[df["inhalative toxicity C2C assessment"] == "GREY", "conc_hom_mat"].sum()
        sum_dermal_grey = df.loc[df["dermal toxicity C2C assessment"] == "GREY", "conc_hom_mat"].sum()

        if float(sum_oral_grey)>=0.001:
            oral_grey.append({
                "hom_material": hom_material,
                "GREY_oral_tox": "Yes"
            })
        else:
            oral_grey.append({
                "hom_material": hom_material,
                "GREY_oral_tox": "No"
            })
        if sum_inhal_grey>=0.001:
            inhal_grey.append({
                "hom_material": hom_material,
                "GREY_inhal_tox": "Yes"
            })
        else:
            inhal_grey.append({
                "hom_material": hom_material,
                "GREY_inhal_tox": "No"
            })
        if sum_dermal_grey>=0.001:
            dermal_grey.append({
                "hom_material": hom_material,
                "GREY_dermal_tox": "Yes"
            })
        else:
            dermal_grey.append({
                "hom_material": hom_material,
                "GREY_dermal_tox": "No"
            })

    oral_grey_df = pd.DataFrame(oral_grey)
    inhal_grey_df = pd.DataFrame(inhal_grey)
    dermal_grey_df = pd.DataFrame(dermal_grey)
    grey_df = oral_grey_df.merge(inhal_grey_df, on="hom_material", how="left").merge(dermal_grey_df, on="hom_material", how="left")
    df_with_ate = df_with_ate.merge(grey_df, on="hom_material", how="left")
    # assess oral tox
    if 'ATE_based_on_LD50_oral' in df_with_ate.columns:
        tox_category = "Acute toxicity oral C2C"
        ate_value = 'ATE_based_on_LD50_oral'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 300, tox_category] = "RED"
        df_with_ate.loc[df_with_ate[ate_value].between(300, 2000, inclusive="right"), tox_category] = "YELLOW"
        df_with_ate.loc[df_with_ate[ate_value]>2000, tox_category] = "GREEN"


    # assess dermal tox
    if 'ATE_based_on_LD50_dermal' in df_with_ate.columns:
        tox_category = "Acute toxicity dermal C2C"
        ate_value = 'ATE_based_on_LD50_dermal'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 1000, tox_category] = "RED"
        df_with_ate.loc[df_with_ate[ate_value].between(1000, 2000, inclusive="right"), tox_category] = "YELLOW"
        df_with_ate.loc[df_with_ate[ate_value]>2000, tox_category] = "GREEN"

    # assess inhal tox gases
    if 'ATE_based_on_LC50_gas' in df_with_ate.columns:
        tox_category = "Acute toxicity inhalation (gases) C2C"
        ate_value = 'ATE_based_on_LC50_gas'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 10, tox_category] = "RED"
        df_with_ate.loc[df_with_ate[ate_value].between(10, 20, inclusive="right"), tox_category] = "YELLOW"
        df_with_ate.loc[df_with_ate[ate_value]>20, tox_category] = "GREEN"

    # assess inhal tox vapour
    if 'ATE_based_on_LC50_vapour' in df_with_ate.columns:
        tox_category = "Acute toxicity inhalation (vapour) C2C"
        ate_value = 'ATE_based_on_LC50_vapour'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 10, tox_category] = "RED"
        df_with_ate.loc[df_with_ate[ate_value].between(10, 20, inclusive="right"), tox_category] = "YELLOW"
        df_with_ate.loc[df_with_ate[ate_value]>20, tox_category] = "GREEN"

    # assess inhal tox dust/mist
    if 'ATE_based_on_LC50_dust_mist_aerosol' in df_with_ate.columns:
        tox_category = "Acute toxicity inhalation (dust/mist) C2C"
        ate_value = 'ATE_based_on_LC50_dust_mist_aerosol'

        df_with_ate[tox_category] = None
        df_with_ate.loc[df_with_ate[ate_value]<= 1, tox_category] = "RED"
        df_with_ate.loc[df_with_ate[ate_value].between(1, 5, inclusive="right"), tox_category] = "YELLOW"
        df_with_ate.loc[df_with_ate[ate_value]>5, tox_category] = "GREEN"

    # The assessment of the whole mixture
    df_with_ate["C2C acute toxicity"] = None

    # fill in red first
    df_with_ate.loc[(df_with_ate["Acute toxicity oral C2C"] == "RED") | (df_with_ate["Acute toxicity dermal C2C"] == "RED") |
                    (df_with_ate["Acute toxicity inhalation (gases) C2C"] == "RED") | (df_with_ate["Acute toxicity inhalation (vapour) C2C"] == "RED")
                    | (df_with_ate["Acute toxicity inhalation (dust/mist) C2C"] == "RED"), "C2C acute toxicity"] = "RED"

    # fill in grey
    df_with_ate.loc[df_with_ate["C2C acute toxicity"].isna() & ((df_with_ate["GREY_oral_tox"] == "Yes") | (df_with_ate["GREY_inhal_tox"] == "Yes") |
                    (df_with_ate["GREY_dermal_tox"] == "Yes")), "C2C acute toxicity"] = "GREY"


    # fill in yellow
    df_with_ate.loc[df_with_ate["C2C acute toxicity"].isna() & ((df_with_ate["Acute toxicity oral C2C"] == "YELLOW") | (df_with_ate["Acute toxicity dermal C2C"] == "YELLOW") |
                    (df_with_ate["Acute toxicity inhalation (gases) C2C"] == "YELLOW") | (df_with_ate["Acute toxicity inhalation (vapour) C2C"] == "YELLOW")
                    | (df_with_ate["Acute toxicity inhalation (dust/mist) C2C"] == "YELLOW")), "C2C acute toxicity"] = "YELLOW"

    # fill in green
    df_with_ate.loc[df_with_ate["C2C acute toxicity"].isna() & ((df_with_ate["Acute toxicity oral C2C"] == "GREEN") | (df_with_ate["Acute toxicity dermal C2C"] == "GREEN") |
                    (df_with_ate["Acute toxicity inhalation (gases) C2C"] == "GREEN") | (df_with_ate["Acute toxicity inhalation (vapour) C2C"] == "GREEN")
                    | (df_with_ate["Acute toxicity inhalation (dust/mist) C2C"] == "GREEN")), "C2C acute toxicity"] = "GREEN"

    return df_with_ate
def C2C_acute_toxicity(df_product, df_toxicity_info, ld_lc_to_assess):
    # merge the df for the product with information about the toxicity
    df_calculation = pd.merge(df_product, df_toxicity_info, on="CAS", how="left")

    # get the unique hom materials
    hom_materials = df_product["Homogenous Material"].unique().tolist()

    # calculate the ATE df
    final_df, unknown_LD_class_chemicals = obtain_all_ATE_df(df_calculation, hom_materials, ld_lc_to_assess)
    if unknown_LD_class_chemicals != []:
        print("Chemicals with unknown ATE and not classified:", unknown_LD_class_chemicals)

    # assess the mixture classification
    acute_tox_df = classify_mixture_C2C(final_df, df_product, df_toxicity_info)

    return acute_tox_df